# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors: FAIR² Dataset Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze a clinical dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is specified via a [Croissant schema](https://mlcommons.org/croissant/) URL.

- **Title:** Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution
- **Identifier:** 10.71728/senscience.qs2f-h81p
- **License:** [Open Data Commons Attribution License](https://opendatacommons.org/licenses/by/1-0/)
- **Description:** Tabular dataset of 77 cancer survivors with second primary colorectal cancer. Includes variables such as age, sex, comorbidities, cancer types, treatment history, diagnosis intervals, anatomical distribution, histopathological subtype, presence of distant metastasis, and MSI/MMR status.


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading

Load the metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL (Croissant JSON-LD schema)
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset loaded successfully!")
print(f"\nTitle      : {metadata.name}")
print(f"Identifier : {metadata.identifier}")
print(f"License    : {metadata.license}")
print(f"Description: {metadata.description}")

## 2. Data Overview

List available `RecordSet` entities, their `@id`s, and their fields/columns by `@id`. This step helps identify what data tables and variables are in the dataset.

Below, we enumerate each `RecordSet` and its schema programmatically using the metadata structure.

In [ ]:
# Enumerate all record sets and their associated fields using their @id
record_set_objs = dataset.metadata.get_record_sets()
print(f"Number of record sets: {len(record_set_objs)}")

for rs in record_set_objs:
    print(f"\nRecord set: {rs['@id']}")
    print(f"  Name        : {rs.get('name', 'N/A')}")
    print(f"  Description : {rs.get('description', 'N/A')}")

    if 'field' in rs:
        print('  Fields (by @id):')
        # If only a single field as dict, wrap in list
        field_list = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        for field in field_list:
            fid = field['@id'] if isinstance(field, dict) else field
            print(f"    - {fid}")

## 3. Data Extraction

Extract data from each record set into DataFrames for exploration. This section uses the `@id` values gleaned from the overview above. You can select the record set of interest by its `@id`.

> **Tip:** Replace the example `record_set_id` with the desired `@id` found above.

In [ ]:
# List of record set @id's. Replace with actual @id's found from the overview above.
record_set_ids = [rs['@id'] for rs in dataset.metadata.get_record_sets()]

dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records from: {record_set_id}")
    records_gen = dataset.records(record_set=record_set_id)
    df = pd.DataFrame(records_gen)
    dataframes[record_set_id] = df
    print(f"  - Rows: {len(df)}, Columns: {len(df.columns)}")

# As example, display columns from the first record set (if any)
if record_set_ids:
    example_id = record_set_ids[0]
    print(f"\nColumns for record set {example_id}:")
    print(dataframes[example_id].columns.tolist())
    display(dataframes[example_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps such as filtering records, normalizing numeric fields, removing outliers, or grouping data by key attributes.

We demonstrate with a numeric field and a demographic grouping field—make sure to reference fields by their `@id`.

In [ ]:
# Choose the record set and fields you want to analyze
# Example assumes a numeric field '@id': 'Age_at_Diagnosis', group field '@id': 'sex'

# Please adjust these @id values to match those printed above for this dataset
record_set_id = record_set_ids[0] if record_set_ids else None
numeric_field_id = None
group_field_id = None

if record_set_id:
    df = dataframes[record_set_id]
    # Try to find appropriate field/column names (@id)
    for col in df.columns:
        if 'age' in col.lower() and ('diagnosis' in col.lower() or 'at_' in col.lower()):
            numeric_field_id = col
        if col.lower() in ('sex', 'gender'):
            group_field_id = col

    if numeric_field_id:
        print(f"Numeric field detected for analysis: {numeric_field_id}")
        threshold = 30
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized '{numeric_field_id}' for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouped analysis
        if group_field_id and group_field_id in filtered_df.columns:
            print(f"\nGrouping by '{group_field_id}':")
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].agg(['count', 'mean', 'std'])
            display(grouped)
    else:
        print("No suitable numeric field detected for EDA. Please update 'numeric_field_id' above.")
else:
    print("No record sets available in the dataset.")

## 5. Visualization

Visualize distributions and relationships using `matplotlib` or `seaborn` for the selected fields. Here we plot the distribution of the detected numeric field by group (e.g., Age at Diagnosis by Sex).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id and numeric_field_id:
    plt.figure(figsize=(7, 5))
    if group_field_id and group_field_id in df.columns:
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.title(f"Distribution of {numeric_field_id} by {group_field_id}")
    else:
        sns.histplot(data=df, x=numeric_field_id, bins=10, kde=True)
        plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

- This notebook demonstrated loading and preliminary exploration of the FAIR² clinical colorectal cancer survivors dataset using `mlcroissant`.
- You successfully listed available record sets and variables by `@id`, loaded records into DataFrames, filtered and normalized numeric fields, performed grouped analysis, and created basic visualizations.

**Next steps:** Leverage this workflow for downstream statistical modeling or connecting to other FAIR datasets via Croissant! For more information, see [mlcroissant documentation](https://github.com/mlcommons/croissant).